# 09. BLIP 기반 이미지 캡셔닝

Kaggle Notebook에서 바로 실행할 수 있는 BLIP 이미지 캡셔닝 예제입니다.

- `BlipProcessor`, `BlipForConditionalGeneration` 사용
- 이미지 URL 또는 로컬 이미지 파일 입력 지원
- 여러 이미지에 반복 적용 가능
- BLIP 영어 캡션을 생성한 뒤 한국어로 번역하여 출력

> Kaggle에서 외부 URL/모델 다운로드가 필요하면 Notebook 오른쪽 설정에서 Internet을 켜 주세요.

In [ ]:
# Kaggle 환경에 필요한 패키지를 설치합니다.
# 이미 설치되어 있으면 빠르게 넘어갑니다.
!pip -q install transformers sentencepiece sacremoses

In [ ]:
import os
from pathlib import Path

import pandas as pd
import requests
import torch
from IPython.display import display
from PIL import Image
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BlipForConditionalGeneration,
    BlipProcessor,
)

# GPU가 있으면 GPU를 사용하고, 없으면 CPU를 사용합니다.
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1. BLIP 모델과 Processor 불러오기

In [ ]:
# BLIP 이미지 캡셔닝 모델입니다.
# base 모델은 비교적 가벼워 Kaggle Notebook에서 테스트하기 좋습니다.
caption_model_name = "Salesforce/blip-image-captioning-base"

processor = BlipProcessor.from_pretrained(caption_model_name)
model = BlipForConditionalGeneration.from_pretrained(caption_model_name).to(device)
model.eval()

# 문제 조건의 model.generator() 형태로 호출할 수 있도록 generate 메서드에 별칭을 붙입니다.
# Hugging Face 표준 메서드 이름은 generate()입니다.
model.generator = model.generate

print("BLIP 모델 로딩 완료:", caption_model_name)

## 2. 영어 캡션을 한글로 번역하는 모델 불러오기

BLIP 캡셔닝 모델은 영어 캡션을 생성합니다. 한글 출력이 필요하므로 영어-한국어 번역 모델을 함께 사용합니다.

In [ ]:
# 기존 Helsinki-NLP/opus-mt-en-ko는 유효하지 않은 모델 ID라 오류가 납니다.
# NLLB 모델은 영어(eng_Latn)를 한국어(kor_Hang)로 번역할 수 있습니다.
translation_model_name = "facebook/nllb-200-distilled-600M"

translation_tokenizer = AutoTokenizer.from_pretrained(
    translation_model_name,
    src_lang="eng_Latn",
)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(translation_model_name).to(device)
translation_model.eval()

print("영어-한국어 번역 모델 로딩 완료:", translation_model_name)

## 3. 이미지 입력 준비

`image_inputs` 리스트에 이미지 URL 또는 Kaggle 로컬 이미지 경로를 여러 개 넣으면 됩니다.

- URL 예시: `https://.../image.jpg`
- Kaggle 데이터셋 경로 예시: `/kaggle/input/my-dataset/sample.jpg`
- 업로드 파일 경로 예시: `/kaggle/working/sample.jpg`

In [ ]:
# 테스트용 이미지 URL입니다. 원하는 URL 또는 로컬 파일 경로로 바꿔도 됩니다.
image_inputs = [
    "https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg",
    # "/kaggle/input/your-dataset/your-image.jpg",
]

image_inputs

## 4. 이미지 로딩, 캡션 생성, 한글 번역 함수 만들기

In [ ]:
def load_image(image_source):
    """URL 또는 로컬 파일 경로에서 이미지를 읽어 RGB PIL 이미지로 반환합니다."""
    image_source = str(image_source)

    if image_source.startswith(("http://", "https://")):
        image = Image.open(requests.get(image_source, stream=True).raw)
    else:
        image = Image.open(image_source)

    return image.convert("RGB")


def translate_to_korean(english_caption):
    """영어 캡션을 한국어 문장으로 번역합니다."""
    inputs = translation_tokenizer(
        english_caption,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(device)

    # NLLB는 target language token을 forced_bos_token_id로 지정해야 합니다.
    korean_token_id = translation_tokenizer.convert_tokens_to_ids("kor_Hang")

    with torch.no_grad():
        translated_ids = translation_model.generate(
            **inputs,
            forced_bos_token_id=korean_token_id,
            max_new_tokens=80,
        )

    return translation_tokenizer.decode(translated_ids[0], skip_special_tokens=True)


def generate_caption(image_source, max_new_tokens=40):
    """이미지 하나에 대해 BLIP 캡션을 만들고 한글로 출력할 문자열을 반환합니다."""
    image = load_image(image_source)

    # Processor로 이미지를 모델 입력 텐서로 전처리합니다.
    inputs = processor(images=image, return_tensors="pt").to(device)

    # model.generator()를 사용하여 캡션 토큰을 생성합니다.
    # 내부적으로는 Hugging Face의 generate()와 같은 동작입니다.
    with torch.no_grad():
        output_ids = model.generator(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=5,
        )

    # 생성된 토큰을 문자열 캡션으로 디코딩합니다.
    english_caption = processor.decode(output_ids[0], skip_special_tokens=True)
    korean_caption = translate_to_korean(english_caption)

    return {
        "image": image,
        "source": str(image_source),
        "english_caption": english_caption,
        "korean_caption": korean_caption,
    }

## 5. 여러 이미지에 캡션 생성하기

In [ ]:
results = []

for idx, image_source in enumerate(image_inputs, start=1):
    print(f"\n[{idx}] 이미지 처리 중: {image_source}")
    result = generate_caption(image_source)
    results.append(result)

    display(result["image"].resize((384, 384)))
    print("영어 캡션:", result["english_caption"])
    print("한글 캡션:", result["korean_caption"])

## 6. 결과를 표로 확인하기

In [ ]:
caption_df = pd.DataFrame(
    [
        {
            "이미지": item["source"],
            "영어 캡션": item["english_caption"],
            "한글 캡션": item["korean_caption"],
        }
        for item in results
    ]
)

caption_df

## 7. 로컬 이미지 폴더 전체에 적용하기

Kaggle 데이터셋 폴더에 이미지가 여러 장 있을 때 사용할 수 있는 예시입니다. `local_image_dir` 값을 본인의 데이터셋 경로로 바꿔 실행하세요.

In [ ]:
# 예: local_image_dir = "/kaggle/input/your-dataset/images"
local_image_dir = None

if local_image_dir is not None:
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    local_image_paths = [
        str(path)
        for path in Path(local_image_dir).glob("**/*")
        if path.suffix.lower() in image_extensions
    ]

    print("찾은 이미지 개수:", len(local_image_paths))
    local_results = [generate_caption(path) for path in local_image_paths]

    local_caption_df = pd.DataFrame(
        [
            {
                "이미지": item["source"],
                "영어 캡션": item["english_caption"],
                "한글 캡션": item["korean_caption"],
            }
            for item in local_results
        ]
    )

    display(local_caption_df)